In [ ]:
import pandas as pd
import json
import re

base_path = ""

def clean_and_parse_result(raw):
    try:
        cleaned = raw.strip('"').replace('\\"', '"')
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {}

def load_and_parse(files, atc_code):
    dfs = []
    for f in files:
        df = pd.read_csv(base_path + f)

        # Parse 'result' JSON
        parsed_results = df["result"].apply(clean_and_parse_result)
        result_df = pd.json_normalize(parsed_results)

        # Drop old 'result' and add parsed columns + ATC code tag
        df = df.drop(columns=["result"]).reset_index(drop=True)
        df = pd.concat([df, result_df], axis=1)
        df["atc_code"] = atc_code

        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load and tag each group
c10_df = load_and_parse(["c10_switch_stops.csv"], "a10")

c10_original = pd.read_csv("c10_switches_original.csv").dropna(subset=['anamnesis'])

def combine_dfs(original, df):
    combined = pd.concat([original.reset_index(drop=True), df.drop(columns=['id']).reset_index(drop=True)], axis=1)
    return combined

# Combine each pair
combined_c10 = combine_dfs(c10_original, c10_df)

# Concatenate all together
final_combined_df = combined_c10

# Optional: Check the result
print(final_combined_df.head())

unique_anamnesis_df = final_combined_df.drop_duplicates(subset='anamnesis')

reason_df = unique_anamnesis_df[unique_anamnesis_df['reason_for_stopping'] != ""]
print(len(reason_df))

In [ ]:
c10_drugs = [
    "atorvastatin",
    "atorvastatiini",
    "atorvastatiin",
    "rosuvastatiin",
    "rosuvastatin",
    "rosuvastatini",
    "simvastatin",
    "simvastatiin",
    "simvastatiini",
    "simvastatinin",
    "rosuvastatiini",
    "lipanthyl",
    "ezetrol",
    "lescol",
    "statiin",
    "statiine",
    "statiini",
    "statiinid",
    "statiinravi",
    "statini",
    "statiin"
]





# Make sure all terms are lowercase
c10_drugs = [drug.lower() for drug in c10_drugs]

# Join the list into a single regex pattern (escaped, joined with | for "OR" logic)
pattern = "|".join(re.escape(drug) for drug in c10_drugs)

# Filter drug names that contain any diabetes-related term (case-insensitive)
statin_df = reason_df[reason_df['drug_name'].str.lower().str.contains(pattern, na=False)]

print(len(statin_df))

In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer
import torch
from openai import OpenAI

import json
import csv
import re
import math
import numpy as np
import pandas as pd
import asyncio
import matplotlib.pyplot as plt 

from pydantic import BaseModel
from enum import Enum

from huggingface_hub import login
HF_TOKEN = ""
login(HF_TOKEN)  # This logs you in for the session

import requests
response = requests.get("https://huggingface.co")
print(response.status_code)  # Should print 200 if the connection is successful

model_name = "neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
! nvidia-smi

llm = LLM(model=model_name, device=device, max_model_len=65536, tensor_parallel_size=1, enable_prefix_caching=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Define your classification categories

categories = [
    "Adverse reactions",
    "Treatment success", #
    "Treatment inefficacy", #
    "Contraindication", #
    "Non-medical reasons",
    "Other",
    "Indeterminate"
]

example_reasons = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
    
]

example_labels = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
]

# Set up guided decoding to restrict outputs to your categories
guided_decoding = GuidedDecodingParams(choice=categories)

# Configure sampling parameters with guided decoding
sampling_params = SamplingParams(guided_decoding=guided_decoding)


system_prompt = {
"role": "system", "content":"""You are a helpful assistant that reads Estonian electronic health records written by doctors about statin discontinuation for their patients with high cholesterol levels. For each input, classify the reason why the patient stopped taking the medication.
Your input is the extracted reason for stopping the drug.

Here are the categories you must use:

1. Adverse reactions: Discontinuation due to adverse side effects, allergic reactions, or negative interactions with other medications.
2. Treatment success: Discontinuation due to successful treatment completion or sufficient improvement in health.
3. Treatment inefficacy: Perceived ineffectiveness of the treatment or loss of belief in its efficacy.
4. Contraindication: Discontinuation due to the emergence or discovery of a medical condition or risk factor that makes the continued use of the treatment unsafe or inappropriate (e.g. "vastunäidustatud").
5. Non-medical reasons: Discontinuation due to factors unrelated to the patient's health or the treatment’s medical effects, such as financial constraints, access issues, personal choice, cultural beliefs, or social circumstances.
6. Other: Other medical reasons not explicitly covered by the other categories.
7. Indeterminate: Unclear or unspecified reasons for discontinuation.

Only choose one category per input. If the reason is ambiguous or not clearly stated, select "Indeterminate". Respond only with the category name."""}

messages_template = [system_prompt]

drug_stop_list = statin_df['reason_for_stopping'].tolist()

# drug_stop_list = drug_stop_list[:10]
# anamnesis_list = anamnesis_list[:10]

for reason, label in zip(example_reasons, example_labels):
    user_content = f"Reason for stopping:\n{reason}"
    messages_template.append({"role": "user", "content": user_content})
    messages_template.append({"role": "assistant", "content": label})


messages_list = [
    messages_template + [{"role": "user", "content": "Reason for stopping:\n" + str(reason)}]
    for reason in zip(drug_stop_list)
]

prompts = [
    tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    for messages in messages_list
]

# Create the prompt for classification

# Generate the classification
outputs = llm.generate(
    prompts=prompts,
    sampling_params=sampling_params
)

# Extract and print the classification result
# classification = outputs[0].outputs[0].text.strip()

In [ ]:
classified_categories = [output.outputs[0].text.strip() for output in outputs]

# Build a dataframe
df = pd.DataFrame({
    "Input": drug_stop_list,
    "Category": classified_categories
})

# Save to CSV
df.to_csv("c10_switches_why_stopped.csv", index=False)

# Optional: print path or success message
print("Results saved to c10_switches_why_stopped.csv")